In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import dill
import pickle

from datasets import load_dataset,load_from_disk
from sklearn.model_selection import GroupShuffleSplit

from src.preprocessing import *
from src.utils import tech_aliases

In [ ]:

# dataset = load_dataset("cnamuangtoun/resume-job-description-fit")
dataset=load_from_disk("../data/raw/resume-job-description-fit")

In [ ]:
# dataset.save_to_disk("../data/raw/resume-job-description-fit")

In [ ]:
hf_train_df=dataset['train'].to_pandas()
hf_test_df=dataset['test'].to_pandas()

In [ ]:
hf_train_df.head()

In [ ]:
hf_train_df.info()

In [ ]:
hf_train_df=hf_train_df.drop_duplicates()
hf_test_df=hf_test_df.drop_duplicates()

In [ ]:
hf_train_df['job_description_text'].duplicated().sum()

In [ ]:
print(hf_train_df['label'].value_counts())
print("\nClass %:")
print(hf_train_df['label'].value_counts(normalize=True) * 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

hf_train_df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['blue','red','green'])
axes[0].set_title('Class Distribution (Count)')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')

hf_train_df['label'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',colors=['blue','red','green'])
axes[1].set_title('Class Distribution (%)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

counts = hf_train_df['label'].value_counts()
ratio = counts.max() / counts.min()
print(f"\nImbalance Ratio: {ratio:.2f}x")

In [ ]:
group_size=hf_train_df.groupby('job_description_text').size()

print(group_size.describe())
print("-"*50)
print("Groups with 1 sample:", (group_size == 1).sum())
print("Groups with 2 samples:", (group_size == 2).sum())
print("Groups with 3+ samples:", (group_size >= 3).sum())

In [ ]:
hf_train_df['resume_len'] = hf_train_df['resume_text'].apply(lambda x: len(x.split()))
hf_train_df['jd_len'] = hf_train_df['job_description_text'].apply(lambda x: len(x.split()))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))

sns.kdeplot(data=hf_train_df,x='resume_len',hue='label',ax=axes[0])
axes[0].set_title('Resume Length Distribution')

sns.kdeplot(data=hf_train_df,x='jd_len',hue='label',ax=axes[1])
axes[1].set_title('JD Length Distribution')

plt.tight_layout()
plt.show()

In [ ]:
hf_train_df.groupby('label')[['resume_len', 'jd_len']].mean()

In [ ]:
hf_train_df.drop(columns=['resume_len', 'jd_len'], inplace=True)

In [ ]:
hf_train_df['resume_exp']=hf_train_df['resume_text'].apply(lambda x:extract_experience(x))
hf_train_df['jd_exp']=hf_train_df['job_description_text'].apply(lambda x:extract_experience(x))

hf_test_df['resume_exp']=hf_test_df['resume_text'].apply(lambda x:extract_experience(x))
hf_test_df['jd_exp']=hf_test_df['job_description_text'].apply(lambda x:extract_experience(x))



In [ ]:
hf_train_df[['resume_exp','jd_exp']].head()

In [ ]:
zero_resume_exp=hf_train_df[hf_train_df['resume_exp']==0]
print(len(zero_resume_exp))

In [ ]:
zero_sample=zero_resume_exp['resume_text'].head(5)
for i,text in enumerate(zero_sample):
  print(f" Sample {i+1}:")
  print(text[:400])
  print()

In [ ]:
zero_jd_exp=hf_train_df[hf_train_df['jd_exp']==0]
print(len(zero_jd_exp))

In [ ]:
def clean_text(text):
    text = text.lower()

    # Remove URLs first (before special char removal breaks them)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # Remove emails
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove phone numbers (10+ digits)
    text = re.sub(r'\b\d{10,}\b', ' ', text)

    # Remove markdown markers but KEEP content inside
    text = re.sub(r'\*+|#+', ' ', text)          # ** ## markers
    text = re.sub(r'_{2,}', ' ', text)            # __ underline markdown

    # Remove brackets but KEEP content inside (years/skills valuable)
    text = re.sub(r'[\[\]()]', ' ', text)

    # Remove page numbers
    text = re.sub(r'page \d+', ' ', text)
    

    return text


In [ ]:
hf_train_df['resume_clean']=hf_train_df['resume_text'].apply(lambda x:clean_text(x))
hf_train_df['jd_clean']=hf_train_df['job_description_text'].apply(lambda x:clean_text(x))


hf_test_df['resume_clean']=hf_test_df['resume_text'].apply(lambda x:clean_text(x))
hf_test_df['jd_clean']=hf_test_df['job_description_text'].apply(lambda x:clean_text(x))

In [ ]:
# def further_clean_text(text):
#     # Remove special characters except spaces/./+/#
#     text = re.sub(r'[^a-z0-9\s\+\#\.]', ' ', text)
    
#     # Remove brackets but KEEP content inside (years/skills valuable)
#     text = re.sub(r'[\[\]()]', ' ', text)
        
#     # Remove numbers
#     text = re.sub(r'\d+', ' ', text)

#     #Remove extra whitespace — always LAST
#     text = re.sub(r'\s+', ' ', text).strip()
    
#     return text

In [ ]:
# hf_train_df['resume_clean']=hf_train_df['resume_clean'].apply(lambda x:further_clean_text(x))
# hf_train_df['jd_clean']=hf_train_df['jd_clean'].apply(lambda x:further_clean_text(x))


# hf_test_df['resume_clean']=hf_test_df['resume_clean'].apply(lambda x:further_clean_text(x))
# hf_test_df['jd_clean']=hf_test_df['jd_clean'].apply(lambda x:further_clean_text(x))

In [ ]:
label_map = {
    "No Fit": 0,
    "Potential Fit": 1,
    "Good Fit": 2
}

hf_train_df['label'] = hf_train_df['label'].map(label_map)
hf_test_df['label'] = hf_test_df['label'].map(label_map)

In [ ]:
gss = GroupShuffleSplit(test_size=0.2,random_state=42)

train_idx,val_idx = next(gss.split(hf_train_df,groups=hf_train_df['job_description_text']))

train_df=hf_train_df.iloc[train_idx]
val_df=hf_train_df.iloc[val_idx]

In [ ]:
pickle.dump(hf_train_df,open('../data/cleaned/clean_train_df.pickle','wb'))

pickle.dump(val_df,open('../data/cleaned/clean_val_df.pickle','wb'))

pickle.dump(hf_test_df,open('../data/cleaned/clean_test_df.pickle','wb'))

In [ ]:
skills=pd.read_csv('../data/skills/raw/JobsDatasetProcessed.csv')

In [ ]:
skills.head()

In [ ]:
skills.duplicated().sum()

In [ ]:
skill_set=extract_skills_from_row(skills,"IT Skills")

In [ ]:
all_skills=[key for key in tech_aliases.keys()]
all_skills.extend(skill_set)
all_skills=set(all_skills)

In [ ]:
list(all_skills)[:5]

In [ ]:
pickle.dump(all_skills,open('../data/skills/processed/all_skills.pkl','wb'))
